# From Border Clashes to Capital Attacks in MENA?

This notebook documents the reproducible workflow for my POLI3148 Assignment 1 project. It uses ACLED as the primary dataset and follows the assignment brief by showing the data-cleaning logic, operationalization choices, descriptive statistics, and interactive visualizations that feed into the final report in `docs/index.html`.

## Research question

Has the rise of drones and other remote attacks in the Middle East and North Africa (MENA) been accompanied by a spatial shift from border-proximate clashes toward attacks on capital areas?

## Analytical strategy

- Use the updated ACLED MENA raw file as the primary dataset.
- Keep both `Middle East` and `Northern Africa`, so the final sample covers the full Middle East and North Africa (MENA) scope in the supplied data.
- Remove rows with missing values in the core fields needed for the analysis.
- Define **capital-area remote attacks** as `Explosions/Remote violence` events whose location is either the national capital itself or a sub-location beginning with the capital name.
- Define **border-proximate battles** as `Battles` events located within 50 kilometres of an international land border.
- Compare the two patterns over time, across countries, and on a map.
- Treat the analysis as an empirical test of this expectation, not as a causal test of whether remote attacks caused the shift.

## Note on ACLED's data generating process

According to ACLED's codebook, the fundamental unit of observation is the event. Events are coded to a date and location, include actor and event-type information, and record fatalities as conservative reported estimates rather than verified totals. ACLED also attempts to code locations as precisely as possible and uses geographic precision fields to indicate confidence in the location data.

For this project, the raw data were supplied in the project folder as a Numbers file and then exported to CSV before analysis. The supplied file contains observations from `Middle East` and `Northern Africa`, which I combine into one MENA sample. The project was motivated in part by ACLED's global trends discussion arguing that drones are being used more than ever before, which made it worth testing whether remote warfare in MENA is also becoming more capital-focused.


In [ ]:
import pandas as pd

from project_utils import (
    CAPITALS_BY_COUNTRY,
    REQUIRED_COLUMNS,
    build_country_table,
    build_map_figure,
    build_remote_growth_figure,
    build_share_figure,
    build_spatial_composition_figure,
    load_raw_data,
    prepare_analysis_data,
    summarize_country_patterns,
    summarize_spatial_bucket,
    summarize_yearly,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## Check the raw file and NA handling

This section documents the supplied regional coverage and the direct missing-value removal step used in the final version of the project.


In [ ]:
raw_df = load_raw_data()
raw_df['region'].value_counts()


In [ ]:
mena_raw = raw_df[raw_df['region'].isin(['Middle East', 'Northern Africa'])].copy()
na_report = mena_raw[REQUIRED_COLUMNS].isna().sum().rename_axis('column').reset_index(name='na_count')
na_report


In [ ]:
df = prepare_analysis_data()

summary_snapshot = {
    "Rows before missing-value removal": len(mena_raw),
    "Rows after missing-value removal": len(df),
    "Rows dropped": len(mena_raw) - len(df),
    "Year range": f"{df['year'].min()}-{df['year'].max()}",
    "Countries": df['country'].nunique(),
    "Remote events": int(df['is_remote'].sum()),
    "Capital-area remote events": int(df['capital_remote_event'].sum()),
    "Border-proximate battles": int(df['border_battle_event'].sum()),
}
pd.Series(summary_snapshot)


## How to read the headline numbers

The report focuses on two analytically important subsets of the full event dataset rather than dividing all events into only two categories.

- The cleaned MENA sample contains all retained events after removing rows with missing values in the core fields.
- `Explosions/Remote violence` is a large event type in the full sample.
- `Capital-area remote attacks` are the subset of remote events that occurred in capital cities or capital sub-locations.
- `Border-proximate battles` are the subset of battles located within 50 kilometres of an international land border.

So the remaining events are mostly remote attacks outside capital areas, battles farther away from borders, and violence against civilians.


## Why use a distance-based border proxy?

In the earlier draft of the project, border conflict was identified only when ACLED notes explicitly mentioned a border or cross-border dynamic. That approach was transparent, but it produced too small and too selective a sample because many frontier clashes occur near international borders without the notes literally using the word `border`.

To improve the measure, this revised version uses a spatial rule instead: a **border-proximate battle** is any battle event located within **50 kilometres** of an international land border. This makes the proxy more geographic and less dependent on wording in the notes, while also expanding the sample enough to capture frontier conflict more plausibly.

The threshold choice also affects sample size:

- Using a `25 km` rule gives `22,063` border-proximate battles, or `22.58%` of all battles.
- Using a `50 km` rule would give `34,154` border-proximate battles, or `34.96%` of all battles.

I keep the `50 km` threshold in the main report because the notes-based approach is too narrow, while the `50 km` rule produces a broader and more defensible measure of frontier conflict. The `25 km` version remains a useful conservative comparison showing that the broader spatial pattern is not created only by widening the border zone.


## Country distribution

The MENA sample is highly uneven across countries, so country-level counts are useful context before interpreting the regional figures.


In [ ]:
df['country'].value_counts().rename_axis('country').reset_index(name='events')


## Growth of drone strikes and remote violence

Because the project is motivated by the rise of drones and other forms of remote violence, it is useful to show that temporal pattern directly before moving to the spatial comparison. The chart below plots annual `Air/drone strike` events alongside the total number of `Explosions/Remote violence` events in the cleaned MENA sample.


In [ ]:
yearly = summarize_yearly(df)
yearly.tail(12)


In [ ]:
remote_growth_fig = build_remote_growth_figure(yearly)
remote_growth_fig.show()


## Share comparison series

This section focuses on the two share measures that are most directly relevant to the research question:

- `border_share_of_battles_pct`
- `capital_share_of_remote_pct`

These show whether capital-area remote attacks and border-proximate battles are becoming more or less important relative to their wider event categories.


In [ ]:
share_fig = build_share_figure(yearly)
share_fig.show()


## Spatial composition

This table compares three groups:

- `Capital areas`
- `Border-proximate areas`
- `Other areas`


In [ ]:
spatial_summary = summarize_spatial_bucket(df)
spatial_summary


In [ ]:
composition_fig = build_spatial_composition_figure(df)
composition_fig.show()


## Country-level drivers

The next outputs identify which countries contribute most strongly to capital-area remote attacks and border-proximate battles.


In [ ]:
country_summary = summarize_country_patterns(df)
country_summary


In [ ]:
country_table = build_country_table(df)
country_table


## Event map

The map focuses only on the two focal categories so the spatial contrast is easier to interpret.


In [ ]:
map_fig = build_map_figure(df)
map_fig.show()


## Interpretation

Three patterns stand out from the notebook output.

1. Remote violence is extremely prevalent across the MENA sample.
2. Capital-area remote attacks are important, but still only a small subset of all remote events.
3. Border-proximate battles remain clearly visible, especially in active frontier theatres such as Yemen, Syria, Iraq, Palestine, and Sudan.

These results do not support a clean shift from borders to capitals. They support a layered interpretation of conflict geography in which remote violence expands without displacing border-proximate conflict.
